## **Imports**

In [1]:
import io
import re
import boto3
import pandas as pd
from sqlalchemy import create_engine

## **Configuration**

In [2]:
# --- CONFIGURATION MINIO ---
MINIO_ENDPOINT = "http://127.0.0.1:9000" # S3 aws
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin123"
BUCKET_NAME = "datalake-noaa"

DB_USER = "myuser"
DB_PASSWORD = "mypassword"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "noaa_silver"

# Définition de la dimension temporelle : les 5 dernières années (2021 à 2025)
TARGET_YEARS = [str(year) for year in range(2021, 2026)]

# Connexions
s3_client = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
)
engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

## **Compter les fichiers MinIO**

In [3]:
# 1. Lister les fichiers CSV français (commençant par "07") pour les années ciblées
csv_objects = []
paginator = s3_client.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=BUCKET_NAME):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        # Filtrer par année (dans le chemin) et stations françaises ("07")
        if any(key.startswith(year) for year in TARGET_YEARS):
            if key.endswith(".csv") and "/" in key and key.split("/")[1].startswith("07"):
                csv_objects.append(key)

print(f"{len(csv_objects)} fichiers trouvés pour la période 2021-2025. Téléchargement...")

1027 fichiers trouvés pour la période 2021-2025. Téléchargement...


In [4]:
# 2. Fonctions de parsing spécifiques au format NOAA
def parse_temperature(val):
    """Ex: '+0121,1' -> 12.1 °C (Facteur d'échelle 10, 9999 = manquant)"""
    if pd.isna(val) or not isinstance(val, str):
        return None
    match = re.match(r"^([+-]?\d+)", val)
    if match:
        num = int(match.group(1))
        if num == 9999:
            return None
        return num / 10.0
    return None

def parse_wind(val):
    """Ex: '270,1,N,0050,1' -> Direction (270°), Vitesse (5.0 m/s)"""
    if pd.isna(val) or not isinstance(val, str):
        return None, None
    parts = val.split(",")
    try:
        direction = float(parts[0]) if parts[0] != "999" else None
        speed = float(parts[3]) / 10.0 if len(parts) > 3 and parts[3] != "9999" else None
        return direction, speed
    except (ValueError, IndexError):
        return None, None

def parse_pressure(val):
    """Ex: '10132,1' -> 1013.2 hPa (Facteur d'échelle 10)"""
    if pd.isna(val) or not isinstance(val, str):
        return None
    match = re.match(r"^(\d+)", val)
    if match:
        num = int(match.group(1))
        if num == 99999:
            return None
        return num / 10.0
    return None

def parse_visibility(val):
    """Ex: '001600,1,N,9' -> Distance en mètres"""
    if pd.isna(val) or not isinstance(val, str):
        return None
    parts = val.split(",")
    try:
        dist = float(parts[0])
        if dist == 999999:
            return None
        return dist
    except (ValueError, IndexError):
        return None

def parse_precipitation(val):
    """Extraction de la hauteur de précipitation (AA1) en millimètres"""
    if pd.isna(val) or not isinstance(val, str):
        return 0.0
    parts = val.split(",")
    try:
        # Le deuxième élément contient généralement la hauteur en dixièmes de mm
        depth = float(parts[1]) / 10.0 if len(parts) > 1 and parts[1] != "9999" else 0.0
        return depth
    except (ValueError, IndexError):
        return 0.0

## **Téléchargement & fusion des fichiers**

In [5]:
# --- 2. TRAITEMENT FICHIER PAR FICHIER (ANTI-CRASH RAM) ---
colonnes_source = ["STATION", "DATE", "TMP", "WND", "SLP", "VIS", "AA1"]
table_name = "silver_meteo_noaa_clean"

first_chunk = True
total_rows_inserted = 0

for idx, key in enumerate(csv_objects):
    try:
        response = s3_client.get_object(Bucket=BUCKET_NAME, Key=key)
        df_temp = pd.read_csv(io.BytesIO(response["Body"].read()), low_memory=False, dtype=str)
        
        cols_presentes = [c for c in colonnes_source if c in df_temp.columns]
        if "STATION" not in cols_presentes or "DATE" not in cols_presentes:
            continue
            
        df_temp = df_temp[cols_presentes]
        
        # Nettoyage unitaire immédiat
        df_clean = pd.DataFrame()
        df_clean["station_id"] = df_temp["STATION"].astype(str)
        df_clean["timestamp"] = pd.to_datetime(df_temp["DATE"], errors="coerce")
        df_clean["temperature"] = df_temp["TMP"].apply(parse_temperature).astype("float64")
        
        if "WND" in df_temp:
            parsed_wind = df_temp["WND"].apply(parse_wind)
            df_clean["wind_direction"] = [x[0] for x in parsed_wind]
            df_clean["wind_speed"] = [x[1] for x in parsed_wind]
        else:
            df_clean["wind_direction"] = None
            df_clean["wind_speed"] = None

        df_clean["wind_direction"] = df_clean["wind_direction"].astype("float64")
        df_clean["wind_speed"] = df_clean["wind_speed"].astype("float64")

        df_clean["pressure"] = df_temp["SLP"].apply(parse_pressure).astype("float64") if "SLP" in df_temp else None
        df_clean["visibility"] = df_temp["VIS"].apply(parse_visibility).astype("float64") if "VIS" in df_temp else None
        df_clean["precipitation"] = df_temp["AA1"].apply(parse_precipitation).astype("float64") if "AA1" in df_temp else 0.0

        # Filtrage qualité rigoureux
        df_clean = df_clean.dropna(subset=["station_id", "timestamp"])
        df_clean = df_clean.drop_duplicates()
        df_clean["precipitation"] = df_clean["precipitation"].fillna(0.0)

        if len(df_clean) == 0:
            continue

        # Envoi sécurisé dans PostgreSQL avec gestion de la connexion
        try:
            mode_ecriture = "replace" if first_chunk else "append"
            df_clean.to_sql(
                name=table_name,
                con=engine,
                if_exists=mode_ecriture,
                index=False,
                chunksize=2000  # Des lots plus petits pour éviter de saturer le pipeline
            )
            first_chunk = False
            total_rows_inserted += len(df_clean)
            
        except Exception as db_err:
            print(f"Erreur d'insertion SQL sur le fichier {key}: {db_err}")
            continue        

        # Envoi progressif dans PostgreSQL (création de la table au 1er fichier, puis ajout)
        mode_ecriture = "replace" if first_chunk else "append"
        df_clean.to_sql(
            name=table_name,
            con=engine,
            if_exists=mode_ecriture,
            index=False,
            chunksize=5000
        )
        
        first_chunk = False
        total_rows_inserted += len(df_clean)

        # Affichage d'un suivi de progression tous les 50 fichiers
        if (idx + 1) % 50 == 0:
            print(f"Progression : {idx + 1}/{len(csv_objects)} fichiers traités ({total_rows_inserted} lignes insérées)...")

    except Exception:
        # Ignore les fichiers corrompus ou erreurs ponctuelles
        continue

print(f"\nTerminé avec succès ! {total_rows_inserted} lignes au total ont été insérées dans PostgreSQL.")

Progression : 50/1027 fichiers traités (1483256 lignes insérées)...
Progression : 100/1027 fichiers traités (3213472 lignes insérées)...
Progression : 150/1027 fichiers traités (4772340 lignes insérées)...
Progression : 200/1027 fichiers traités (6190874 lignes insérées)...
Progression : 250/1027 fichiers traités (7723870 lignes insérées)...
Progression : 300/1027 fichiers traités (9539934 lignes insérées)...
Progression : 350/1027 fichiers traités (11200818 lignes insérées)...
Progression : 400/1027 fichiers traités (12696874 lignes insérées)...
Progression : 450/1027 fichiers traités (14205514 lignes insérées)...
Progression : 500/1027 fichiers traités (15974636 lignes insérées)...
Progression : 550/1027 fichiers traités (17697234 lignes insérées)...
Progression : 600/1027 fichiers traités (19112742 lignes insérées)...
Progression : 650/1027 fichiers traités (20534772 lignes insérées)...
Progression : 700/1027 fichiers traités (22189360 lignes insérées)...
Progression : 750/1027 fich